# LangChain Runnable Graph Reference

Developer-facing top-level statements defined in `langchain_core.runnables.graph`.

# `Stringifiable: Protocol`

`Stringifiable` represents an object that can be converted into text using `str()`.

## Required Method

### `__str__`

```python
__str__(
    self, # Current object
) -> str # Return the string representation
```

---

# `LabelsDict: TypedDict`

`LabelsDict` stores replacement labels used when rendering graph nodes and edges.

## Fields

```python
nodes: dict[str, str] # Original node identifiers or names mapped to displayed labels
edges: dict[str, str] # Original edge labels mapped to displayed labels
```

---

# `is_uuid`

Checks whether a string is a valid UUID.

```python
is_uuid(
    value: str, # String checked as a UUID
) -> bool # Return True when the string is a valid UUID
```

---

# `Edge: NamedTuple`

`Edge` represents one directed connection between two graph nodes.

## Fields and Constructor

```python
Edge(
    source: str, # Identifier of the source node
    target: str, # Identifier of the target node
    data: Stringifiable | None = None, # Optional edge label or associated data
    conditional: bool = False, # Whether the edge represents conditional routing
) # Create a directed graph edge
```

## Method

### `copy`

Returns a new edge with an optionally replaced source or target.

```python
copy(
    self, # Current edge
    *,
    source: str | None = None, # Optional replacement source identifier
    target: str | None = None, # Optional replacement target identifier
) -> Edge # Return the copied edge
```

The original `data` and `conditional` values are preserved.

---

# `Node: NamedTuple`

`Node` represents one item in a Runnable graph.

## Fields and Constructor

```python
Node(
    id: str, # Unique node identifier
    name: str, # Human-readable node name
    data: TypeBaseModel | Runnable[Any, Any] | None, # Runnable, Pydantic schema, or no associated data
    metadata: dict[str, Any] | None, # Optional node metadata
) # Create a graph node
```

## Method

### `copy`

Returns a new node with an optionally replaced identifier or name.

```python
copy(
    self, # Current node
    *,
    id: str | None = None, # Optional replacement node identifier
    name: str | None = None, # Optional replacement node name
) -> Node # Return the copied node
```

The original `data` and `metadata` values are preserved.

---

# `Branch: NamedTuple`

`Branch` stores graph information describing a conditional branch.

## Fields and Constructor

```python
Branch(
    condition: Callable[..., str], # Callable returning a string representation of the condition
    ends: dict[str, str] | None, # Optional branch-result keys mapped to destination node identifiers
) # Create conditional branch metadata
```

---

# `CurveStyle: Enum`

`CurveStyle` defines the edge-curve styles supported by Mermaid rendering.

```python
CurveStyle.BASIS # Basis spline
CurveStyle.BUMP_X # Horizontal bump curve
CurveStyle.BUMP_Y # Vertical bump curve
CurveStyle.CARDINAL # Cardinal spline
CurveStyle.CATMULL_ROM # Catmull-Rom spline
CurveStyle.LINEAR # Straight line
CurveStyle.MONOTONE_X # Monotone curve along the x-axis
CurveStyle.MONOTONE_Y # Monotone curve along the y-axis
CurveStyle.NATURAL # Natural spline
CurveStyle.STEP # Step curve
CurveStyle.STEP_AFTER # Step curve changing after each point
CurveStyle.STEP_BEFORE # Step curve changing before each point
```

---

# `NodeStyles: dataclass`

`NodeStyles` stores Mermaid style declarations for ordinary, first, and last nodes.

## Constructor and Fields

```python
NodeStyles(
    default: str = "fill:#f2f0ff,line-height:1.2", # Style applied to ordinary nodes
    first: str = "fill-opacity:0", # Style applied to the first node
    last: str = "fill:#bfb6fc", # Style applied to the last node
) # Create the Mermaid node-style configuration
```

---

# `MermaidDrawMethod: Enum`

`MermaidDrawMethod` selects how Mermaid syntax is rendered into an image.

```python
MermaidDrawMethod.PYPPETEER # Render locally using a headless Pyppeteer browser
MermaidDrawMethod.API # Render remotely using the Mermaid.INK API
```

---

# `node_data_str`

Returns a readable name for node data.

```python
node_data_str(
    id: str, # Node identifier
    data: TypeBaseModel | Runnable[Any, Any] | None, # Runnable, Pydantic schema, or no node data
) -> str # Return the readable node-data name
```

## Behaviour

- Returns `id` when it is not a UUID or when `data` is `None`.
- Uses `Runnable.get_name()` for Runnable data.
- Uses the class name for Pydantic schema data.
- Removes a leading `Runnable` prefix from generated Runnable names.

---

# `node_data_json`

Converts a node's data and metadata into a JSON-serializable dictionary.

```python
node_data_json(
    node: Node, # Node converted into JSON-compatible data
    *,
    with_schemas: bool = False, # Whether complete Pydantic JSON schemas are included
) -> dict[str, str | dict[str, Any]] # Return the serialized node-data representation
```

## Behaviour

- Marks serializable and non-serializable Runnables as `"runnable"`.
- Marks Pydantic model classes as `"schema"`.
- Includes a complete JSON schema when `with_schemas=True`.
- Marks unsupported data as `"unknown"`.
- Adds node metadata when present.

---

# `Graph: dataclass`

`Graph` stores the nodes and directed edges used to represent a Runnable's structure.

## Fields and Constructor

```python
Graph(
    nodes: dict[str, Node] = {}, # Nodes mapped by their unique identifiers
    edges: list[Edge] = [], # Directed edges connecting graph nodes
) # Create a Runnable graph
```

The actual defaults are created with independent `default_factory` values.

## Methods

### `to_json`

Converts the graph into a JSON-serializable representation.

```python
to_json(
    self, # Current graph
    *,
    with_schemas: bool = False, # Whether complete Pydantic schemas are included
) -> dict[str, list[dict[str, Any]]] # Return serialized nodes and edges
```

UUID node identifiers are replaced by stable numeric identifiers in the returned representation.

### `__bool__`

Returns whether the graph contains at least one node.

### `next_id`

Generates a new unique hexadecimal node identifier.

```python
next_id(
    self, # Current graph
) -> str # Return a new unique node identifier
```

### `add_node`

Adds one node to the graph.

```python
add_node(
    self, # Current graph
    data: TypeBaseModel | Runnable[Any, Any] | None, # Runnable, schema, or no associated node data
    id: str | None = None, # Optional explicit node identifier
    *,
    metadata: dict[str, Any] | None = None, # Optional node metadata
) -> Node # Return the added node
```

A unique identifier is generated when `id` is omitted.

Raises `ValueError` when the supplied identifier already exists.

### `remove_node`

Removes one node and every edge connected to it.

```python
remove_node(
    self, # Current graph
    node: Node, # Node removed from the graph
) -> None # Remove the node and its connected edges
```

### `add_edge`

Adds one directed edge between existing nodes.

```python
add_edge(
    self, # Current graph
    source: Node, # Existing source node
    target: Node, # Existing target node
    data: Stringifiable | None = None, # Optional edge label or associated data
    conditional: bool = False, # Whether the edge represents conditional routing
) -> Edge # Return the added edge
```

Raises `ValueError` when the source or target does not belong to the graph.

### `extend`

Adds all nodes and edges from another graph.

```python
extend(
    self, # Graph receiving the additional subgraph
    graph: Graph, # Graph whose nodes and edges are copied
    *,
    prefix: str = "", # Optional prefix added to readable node identifiers
) -> tuple[Node | None, Node | None] # Return the copied first and last nodes
```

The graphs are not automatically connected.

The prefix is omitted when all incoming node identifiers are UUIDs.

### `reid`

Returns a new graph with readable unique identifiers where possible.

```python
reid(
    self, # Current graph
) -> Graph # Return the graph with readable node identifiers
```

Duplicate readable names receive numeric suffixes.

Non-UUID identifiers are preserved.

### `first_node`

Returns the single node that is not the target of any edge.

```python
first_node(
    self, # Current graph
) -> Node | None # Return the unique first node or None
```

Returns `None` when no candidate exists or when multiple candidates exist.

### `last_node`

Returns the single node that is not the source of any edge.

```python
last_node(
    self, # Current graph
) -> Node | None # Return the unique last node or None
```

Returns `None` when no candidate exists or when multiple candidates exist.

### `trim_first_node`

Removes the first node when it has one outgoing edge and removing it still leaves a valid unique first node.

```python
trim_first_node(
    self, # Current graph
) -> None # Conditionally remove the first node
```

### `trim_last_node`

Removes the last node when it has one incoming edge and removing it still leaves a valid unique last node.

```python
trim_last_node(
    self, # Current graph
) -> None # Conditionally remove the last node
```

### `draw_ascii`

Renders the graph as multiline ASCII art.

```python
draw_ascii(
    self, # Current graph
) -> str # Return the ASCII graph
```

Requires the optional `grandalf` dependency.

### `print_ascii`

Prints the ASCII graph directly.

```python
print_ascii(
    self, # Current graph
) -> None # Print the ASCII graph
```

### `draw_png`

Renders the graph as a PNG using Graphviz and `pygraphviz`.

```python
draw_png(
    self, # Current graph
    output_file_path: str | None = None, # Optional file path used to save the PNG
    fontname: str | None = None, # Optional font used for node and edge labels
    labels: LabelsDict | None = None, # Optional replacement labels
) -> bytes | None # Return PNG bytes when no path is supplied; otherwise return None
```

### `draw_mermaid`

Converts the graph into Mermaid flowchart syntax.

```python
draw_mermaid(
    self, # Current graph
    *,
    with_styles: bool = True, # Whether Mermaid frontmatter and class styles are included
    curve_style: CurveStyle = CurveStyle.LINEAR, # Curve style used for graph edges
    node_colors: NodeStyles | None = None, # Optional styles for ordinary, first, and last nodes
    wrap_label_n_words: int = 9, # Number of words placed on each wrapped label line
    frontmatter_config: dict[str, Any] | None = None, # Optional Mermaid frontmatter configuration
) -> str # Return Mermaid flowchart syntax
```

The graph is re-identified before Mermaid syntax is generated.

### `draw_mermaid_png`

Renders the graph as PNG bytes through Mermaid.

```python
draw_mermaid_png(
    self, # Current graph
    *,
    curve_style: CurveStyle = CurveStyle.LINEAR, # Curve style used for graph edges
    node_colors: NodeStyles | None = None, # Optional styles for ordinary, first, and last nodes
    wrap_label_n_words: int = 9, # Number of words placed on each wrapped label line
    output_file_path: str | None = None, # Optional file path used to save the PNG
    draw_method: MermaidDrawMethod = MermaidDrawMethod.API, # Mermaid rendering method
    background_color: str = "white", # Background colour name or hexadecimal colour
    padding: int = 10, # Padding around locally rendered graph content
    max_retries: int = 1, # Maximum API retry count after the initial request
    retry_delay: float = 1.0, # Base delay used for API retry backoff
    frontmatter_config: dict[str, Any] | None = None, # Optional Mermaid frontmatter configuration
    base_url: str | None = None, # Optional custom Mermaid rendering API URL
    proxies: dict[str, str] | None = None, # Optional HTTP and HTTPS proxy configuration
) -> bytes # Return the rendered PNG bytes
```

## Rendering Dependencies

```bash
pip install grandalf # Enable ASCII graph rendering
pip install pygraphviz # Enable Graphviz PNG rendering
pip install requests # Enable Mermaid API rendering
pip install pyppeteer # Enable local Mermaid browser rendering
```

## Developer-Facing Top-Level Statements

```python
Stringifiable # Protocol for objects supporting string conversion
LabelsDict # Node and edge label mapping
is_uuid # UUID validation function
Edge # Directed graph edge
Node # Graph node
Branch # Conditional branch metadata
CurveStyle # Mermaid curve-style enumeration
NodeStyles # Mermaid node-style configuration
MermaidDrawMethod # Mermaid PNG rendering method
node_data_str # Readable node-data name helper
node_data_json # JSON-compatible node-data helper
Graph # Runnable graph container and renderer
```

In [1]:
from langchain_core.runnables.graph import Graph # Import the Graph class

graph: Graph = Graph() # Create an empty graph

input_node = graph.add_node( # Add the first node
    data=None, # Store no Runnable or schema data
    id="input", # Assign a unique node identifier
    metadata={"role": "start"}, # Attach optional metadata
) # Finish creating the input node

process_node = graph.add_node( # Add the processing node
    data=None, # Store no Runnable or schema data
    id="process", # Assign a unique node identifier
    metadata={"operation": "uppercase"}, # Attach processing metadata
) # Finish creating the processing node

output_node = graph.add_node( # Add the final node
    data=None, # Store no Runnable or schema data
    id="output", # Assign a unique node identifier
    metadata={"role": "end"}, # Attach optional metadata
) # Finish creating the output node

first_edge = graph.add_edge( # Add an edge from input to process
    source=input_node, # Set the source node
    target=process_node, # Set the target node
    data="send text", # Add an edge label
    conditional=False, # Mark it as a normal edge
) # Finish creating the first edge

second_edge = graph.add_edge( # Add an edge from process to output
    source=process_node, # Set the source node
    target=output_node, # Set the target node
    data="return result", # Add an edge label
    conditional=False, # Mark it as a normal edge
) # Finish creating the second edge

first_node = graph.first_node() # Find the unique starting node

last_node = graph.last_node() # Find the unique ending node

graph_json: dict = graph.to_json() # Convert the graph into JSON-compatible data

mermaid_code: str = graph.draw_mermaid() # Convert the graph into Mermaid syntax

print("First node:", first_node.name if first_node else None) # Display the first node name

print("Last node:", last_node.name if last_node else None) # Display the last node name

print("Graph JSON:", graph_json) # Display the serialized graph

print(mermaid_code) # Display the Mermaid graph syntax

First node: input
Last node: output
Graph JSON: {'nodes': [{'id': 'input', 'metadata': {'role': 'start'}}, {'id': 'process', 'metadata': {'operation': 'uppercase'}}, {'id': 'output', 'metadata': {'role': 'end'}}], 'edges': [{'source': 'input', 'target': 'process', 'data': 'send text'}, {'source': 'process', 'target': 'output', 'data': 'return result'}]}
---
config:
  flowchart:
    curve: linear
---
graph TD;
	input([input<hr/><small><em>role = start</em></small>]):::first
	process(process<hr/><small><em>operation = uppercase</em></small>)
	output([output<hr/><small><em>role = end</em></small>]):::last
	input -- &nbsp;send text&nbsp; --> process;
	process -- &nbsp;return result&nbsp; --> output;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

